<a href="https://colab.research.google.com/github/joaocm1804/projeto-integracao-eq9/blob/main/00_exploracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Exploração dos dados da Febre de Chikungunya (SINAN)

Antes de construir os pipelines ETL e ELT, vamos conhecer as bases de 2023, 2024 e 2025. Verificamos
estrutura, valores ausentes, códigos e inconsistências.

**Fonte:** https://dadosabertos.saude.gov.br/dataset/arboviroses-febre-de-chikungunya

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Mostra todas as colunas nas tabelas (a base tem 122)
pd.set_option("display.max_columns", 130)
pd.set_option("display.max_rows", 200)

# Conecta o Google Drive ao Colab
from google.colab import drive
drive.mount("/content/drive")

# Pasta onde estão os CSVs no seu Drive
DATA_DIR = "/content/drive/MyDrive/projeto_integracao/dados"

print(os.listdir(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['CHIKBR23.csv', 'CHIKBR24.csv', 'CHIKBR25.csv', 'dic_dados_chikungunya.pdf']


In [ ]:
ARQUIVOS = {
    2023: "CHIKBR23.csv",
    2024: "CHIKBR24.csv",
    2025: "CHIKBR25.csv",
}

dfs = {}
for ano, arquivo in ARQUIVOS.items():
    caminho = os.path.join(DATA_DIR, arquivo)
    dfs[ano] = pd.read_csv(caminho, dtype=str, low_memory=False)
    print(f"{ano}: {len(dfs[ano]):,} linhas x {dfs[ano].shape[1]} colunas")

2023: 266,442 linhas x 122 colunas
2024: 426,214 linhas x 122 colunas
2025: 253,753 linhas x 122 colunas


## Estrutura: os três anos têm as mesmas colunas?
Se as colunas forem iguais e na mesma ordem, a unificação dos anos pode ser um simples empilhamento
(`pd.concat` no ETL, `UNION ALL` no ELT).

In [ ]:
colunas_2023 = list(dfs[2023].columns)

for ano, df in dfs.items():
    iguais = list(df.columns) == colunas_2023
    print(f"{ano}: {df.shape[1]} colunas | mesma estrutura de 2023? {iguais}")

print("\nColunas:", ", ".join(colunas_2023))

2023: 122 colunas | mesma estrutura de 2023? True
2024: 122 colunas | mesma estrutura de 2023? True
2025: 122 colunas | mesma estrutura de 2023? True

Colunas: TP_NOT, ID_AGRAVO, DT_NOTIFIC, SEM_NOT, NU_ANO, SG_UF_NOT, ID_MUNICIP, ID_REGIONA, ID_UNIDADE, DT_SIN_PRI, SEM_PRI, ANO_NASC, NU_IDADE_N, CS_SEXO, CS_GESTANT, CS_RACA, CS_ESCOL_N, SG_UF, ID_MN_RESI, ID_RG_RESI, ID_PAIS, DT_INVEST, ID_OCUPA_N, FEBRE, MIALGIA, CEFALEIA, EXANTEMA, VOMITO, NAUSEA, DOR_COSTAS, CONJUNTVIT, ARTRITE, ARTRALGIA, PETEQUIA_N, LEUCOPENIA, LACO, DOR_RETRO, DIABETES, HEMATOLOG, HEPATOPAT, RENAL, HIPERTENSA, ACIDO_PEPT, AUTO_IMUNE, DT_CHIK_S1, DT_CHIK_S2, DT_PRNT, RES_CHIKS1, RES_CHIKS2, RESUL_PRNT, DT_SORO, RESUL_SORO, DT_NS1, RESUL_NS1, DT_VIRAL, RESUL_VI_N, DT_PCR, RESUL_PCR_, SOROTIPO, HISTOPA_N, IMUNOH_N, HOSPITALIZ, DT_INTERNA, UF, MUNICIPIO, TPAUTOCTO, COUFINF, COPAISINF, COMUNINF, CLASSI_FIN, CRITERIO, DOENCA_TRA, CLINC_CHIK, EVOLUCAO, DT_OBITO, DT_ENCERRA, ALRM_HIPOT, ALRM_PLAQ, ALRM_VOM, ALRM_SANG, A

## Valores ausentes por coluna e por ano
Calculamos o percentual de nulos de cada coluna em cada ano para identificar:
- colunas quase vazias em todos os anos → candidatas a descarte;
- colunas cujo preenchimento muda muito entre os anos → mudança na coleta dos dados.

In [ ]:
nulos = pd.DataFrame({ano: df.isna().mean() * 100 for ano, df in dfs.items()}).round(1)
nulos.columns = [f"% nulo {a}" for a in nulos.columns]

quase_vazias = nulos[(nulos > 95).all(axis=1)]
print(f"{len(quase_vazias)} colunas com mais de 95% de nulos em TODOS os anos:")
quase_vazias

9 colunas com mais de 95% de nulos em TODOS os anos:


,% nulo 2023,% nulo 2024,% nulo 2025
DT_CHIK_S2,99.4,99.5,100.0
DT_PRNT,97.9,97.1,100.0
DT_SORO,99.7,100.0,100.0
DT_NS1,99.9,100.0,100.0
DT_VIRAL,99.3,99.2,99.2
DT_INTERNA,97.6,97.4,96.2
DT_OBITO,99.8,99.8,99.7
DT_ALRM,100.0,100.0,100.0
DT_GRAV,100.0,100.0,100.0


In [ ]:
variacao = nulos.max(axis=1) - nulos.min(axis=1)
mudou = nulos[variacao > 20].assign(**{"variação (p.p.)": variacao[variacao > 20]})
print("Colunas cujo % de nulos varia mais de 20 pontos percentuais entre os anos:")
mudou.sort_values("variação (p.p.)", ascending=False)

Colunas cujo % de nulos varia mais de 20 pontos percentuais entre os anos:


,% nulo 2023,% nulo 2024,% nulo 2025,variação (p.p.)
COPAISINF,69.4,59.9,0.0,69.4
RES_CHIKS1,48.8,47.8,97.6,49.8
RESUL_PRNT,62.3,63.4,97.6,35.3
RES_CHIKS2,62.6,64.4,97.6,35.0
DT_CHIK_S1,72.2,68.0,98.8,30.8


In [ ]:
nulos.assign(media=nulos.mean(axis=1)).sort_values("media", ascending=False)

,% nulo 2023,% nulo 2024,% nulo 2025,media
DT_GRAV,100.0,100.0,100.0,100.000000
DT_ALRM,100.0,100.0,100.0,100.000000
DT_NS1,99.9,100.0,100.0,99.966667
DT_SORO,99.7,100.0,100.0,99.900000
DT_OBITO,99.8,99.8,99.7,99.766667
DT_CHIK_S2,99.4,99.5,100.0,99.633333
DT_VIRAL,99.3,99.2,99.2,99.233333
DT_PRNT,97.9,97.1,100.0,98.333333
LACO_N,100.0,94.6,97.6,97.400000
MIGRADO_W,100.0,94.6,97.6,97.400000


## Campos codificados × dicionário de dados
O SINAN guarda quase tudo como código. Vamos ver a distribuição de cada campo nos três anos lado a lado
e comparar com os valores que o dicionário (`dic_dados_chikungunya.pdf`) diz serem válidos.

In [ ]:
def distribuicao(coluna, top=15):
    """Contagem de valores (incluindo nulos) de uma coluna, um ano por coluna."""
    return (
        pd.DataFrame({ano: df[coluna].value_counts(dropna=False) for ano, df in dfs.items()})
        .fillna(0).astype(int)
        .head(top)
    )

### Perfil do paciente
Valores válidos segundo o dicionário:
- `CS_SEXO`: M masculino, F feminino, I ignorado
- `CS_RACA`: 1 branca, 2 preta, 3 amarela, 4 parda, 5 indígena, 9 ignorado
- `CS_GESTANT`: 1–3 trimestre, 4 idade gestacional ignorada, 5 não, 6 não se aplica, 9 ignorado
- `CS_ESCOL_N`: 0 analfabeto … 8 superior completo, 9 ignorado, 10 não se aplica

In [ ]:
for c in ["CS_SEXO", "CS_RACA", "CS_GESTANT", "CS_ESCOL_N"]:
    print(f"\n=== {c} ===")
    display(distribuicao(c))


=== CS_SEXO ===


,2023,2024,2025
CS_SEXO,,,
F,156668,251700,148482
I,483,392,161
M,109291,174120,105110
NaN,0,2,0



=== CS_RACA ===


,2023,2024,2025
CS_RACA,,,
1,47056,103006,87118
2,13612,24327,12308
3,4377,6899,4518
4,159285,237057,134722
5,931,2034,1365
9,41181,52890,13722
NaN,0,1,0



=== CS_GESTANT ===


,2023,2024,2025
CS_GESTANT,,,
6,135121,215452,129267
5,100527,167733,107762
9,26763,36927,12163
2,1328,1970,1521
3,1202,1903,1452
1,830,1380,1054
4,610,806,505
NaN,61,43,29



=== CS_ESCOL_N ===


,2023,2024,2025
CS_ESCOL_N,,,
0,1804,2400,1498
05,0,0,1
07,0,0,1
1,9096,15001,10155
10,15147,23331,16990
2,4983,8831,6944
3,15031,24043,18144
4,9474,18466,12729
5,12343,20866,15592


### Desfecho do caso
Valores válidos segundo o dicionário:
- `CLASSI_FIN`: 1 confirmado, 2 descartado, 8 inconclusivo
- `CRITERIO`: 1 laboratório, 2 clínico-epidemiológico, 3 clínico
- `EVOLUCAO`: 1 cura, 2 óbito pelo agravo, 3 óbito por outras causas, 4 óbito em investigação, 9 ignorado
- `HOSPITALIZ`: 1 sim, 2 não, 9 ignorado
- `CLINC_CHIK`: 1 aguda, 2 crônica

In [ ]:
for c in ["CLASSI_FIN", "CRITERIO", "EVOLUCAO", "HOSPITALIZ", "CLINC_CHIK"]:
    print(f"\n=== {c} ===")
    display(distribuicao(c))


=== CLASSI_FIN ===


,2023,2024,2025
CLASSI_FIN,,,
0,385,0,48
13,130196,232047,110477
5,108044,162176,127054
8,0,0,1
NaN,27817,31991,16173



=== CRITERIO ===


,2023,2024,2025
CRITERIO,,,
0,308,131,67
1,114584,215193,149357
2,122048,175841,86689
3,3743,6062,3976
NaN,25759,28987,13664



=== EVOLUCAO ===


,2023,2024,2025
EVOLUCAO,,,
0,1238,459,214
1,200669,347598,210610
2,124,247,131
3,372,653,503
4,46,69,39
9,22151,24564,12670
NaN,41842,52624,29586



=== HOSPITALIZ ===


,2023,2024,2025
HOSPITALIZ,,,
0,2,0,0
1,7480,12655,10943
2,162176,291030,189112
9,6387,7787,1580
NaN,90397,114742,52118



=== CLINC_CHIK ===


,2023,2024,2025
CLINC_CHIK,,,
/,3,0,0
0,0,9703,2576
1,156444,276208,152145
2,4842,3839,1773
NaN,105153,136464,97259


### Código do agravo e sistema de origem
Todo registro deveria ter o mesmo agravo: CID-10 **A92.0** (febre de chikungunya).
Verificamos se o formato é igual nos três anos e se alguma variação tem relação com o sistema
que gerou o registro (`TP_SISTEMA`).

In [ ]:
display(distribuicao("ID_AGRAVO"))
print('\n')
display(distribuicao("TP_SISTEMA"))

print("Cruzamento ID_AGRAVO × TP_SISTEMA em 2025:")
pd.crosstab(dfs[2025]["ID_AGRAVO"], dfs[2025]["TP_SISTEMA"].fillna("nulo"))

,2023,2024,2025
ID_AGRAVO,,,
A92.0,266442,426214,247725
A920,0,0,6028


,2023,2024,2025
TP_SISTEMA,,,
/,6,0,0
0,0,22983,6028
2,243076,394445,243190
NaN,23360,8786,4535


Cruzamento ID_AGRAVO × TP_SISTEMA em 2025:


TP_SISTEMA,0,2,nulo
ID_AGRAVO,,,
A92.0,0,243190,4535
A920,6028,0,0


### Quem são os registros com `TP_SISTEMA = 0`?
Hipótese: vêm de um estado que usa sistema próprio e envia os dados ao Ministério num formato diferente
(zeros em vez de vazio, CID sem ponto). Para testar, cruzamos com a UF de notificação.

In [ ]:
UF_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

for ano in [2024, 2025]:
    df = dfs[ano]
    print(f"\n{ano} — UFs dos registros com TP_SISTEMA = 0:")
    display(df.loc[df["TP_SISTEMA"] == "0", "SG_UF_NOT"].map(UF_SIGLA).value_counts())


2024 — UFs dos registros com TP_SISTEMA = 0:


,count
SG_UF_NOT,
ES,22983



2025 — UFs dos registros com TP_SISTEMA = 0:


,count
SG_UF_NOT,
ES,6028


### Resultados de exame laboratorial por UF
Na análise de nulos, o resultado da sorologia (`RES_CHIKS1`) passou de ~48% de nulos em 2023/2024 para ~98% em 2025.
Verificamos se isso é geral ou concentrado em algumas UFs.

In [ ]:
lab = pd.DataFrame({
    ano: df.groupby("SG_UF_NOT")["RES_CHIKS1"].apply(lambda s: round(s.notna().mean() * 100, 1))
    for ano, df in dfs.items()
}).rename(index=UF_SIGLA)
lab.columns = [f"% com resultado {a}" for a in lab.columns]
lab.sort_values(lab.columns[-1], ascending=False)

,% com resultado 2023,% com resultado 2024,% com resultado 2025
SG_UF_NOT,,,
ES,100.0,100.0,100.0
AC,38.8,52.5,0.0
RO,46.4,43.6,0.0
RR,89.1,83.7,0.0
PA,76.3,74.1,0.0
AP,91.1,90.6,0.0
TO,91.8,61.7,0.0
MA,46.4,53.8,0.0
PI,57.4,53.9,0.0
